# Lightkurve Dataset Builder
Downloads Kepler star flux data and saves it as a CSV that matches `exoTest.csv`.

**Columns:** `LABEL` (2=exoplanet, 1=non-planet) | `FLUX.1` ... `FLUX.N`

In [1]:
import lightkurve as lk
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

TEST_CSV = Path('Data/exoTest.csv')
TRAIN_CSV = Path('Data/exoTrain.csv')
OUTPUT_CSV = Path('Data/temp3.csv')
OUTPUT_CSV.parent.mkdir(exist_ok=True)

ref_columns = pd.read_csv(TEST_CSV, nrows=0).columns.tolist()
train_columns = pd.read_csv(TRAIN_CSV, nrows=0).columns.tolist()

N_POINTS = len(ref_columns) - 1                    # flux points per star (pad / truncate)
START = 101
END = 150
print('lightkurve version:', lk.__version__)
print(f'Reference columns from {TEST_CSV.name}: {len(ref_columns)}')
print(f'N_POINTS detected from exoTest: {N_POINTS}')
print(f'exoTrain/exoTest columns identical: {train_columns == ref_columns}')

c:\Code_Space\Machine-Learning\ExoPlanet-Detection\.venv\Lib\site-packages\lightkurve\prf\__init__.py:7: UserWarning: Warning: the tpfmodel submodule is not available without oktopus installed, which requires a current version of autograd. See #1452 for details.
  warnings.warn(


lightkurve version: 2.5.1
Reference columns from exoTest.csv: 3198
N_POINTS detected from exoTest: 3197
exoTrain/exoTest columns identical: True


In [2]:
# Load KOI catalog
koi = pd.read_csv('Data/exo_list.csv')

confirmed  = koi[koi['koi_disposition'] == 'CONFIRMED'][['kepid']].assign(label=2)
false_pos  = koi[koi['koi_disposition'] == 'FALSE POSITIVE'][['kepid']].assign(label=1)

# Combine and sample (start with e.g. 500 balanced)
sample = pd.concat([
    confirmed,
    false_pos
])

STARS = [(f"KIC {row.kepid}", row.label) for _, row in sample.iterrows()]
# Now plug STARS into your existing download loop — it works as-is!
STARS = STARS[START:END]


In [ ]:
# Download, clean, and collect flux for every star
rows = []

for i, (name, label) in enumerate(STARS, 1):
    print(f'[{i}/{len(STARS)}] {name} ...', end=' ')
    try:
        lcc  = lk.search_lightcurve(name, mission='Kepler', author='Kepler', cadence='long').download_all()
        lc   = lcc.stitch().remove_nans()
        # lc   = lcc.stitch().remove_nans().remove_outliers().normalize()
        flux = lc.flux.value.astype(np.float32)

        # pad / truncate to exactly N_POINTS
        if len(flux) >= N_POINTS:
            flux = flux[:N_POINTS]
        else:
            flux = np.concatenate([flux, np.zeros(N_POINTS - len(flux), dtype=np.float32)])

        row = {'LABEL': int(label)}
        row.update({f'FLUX.{j + 1}': float(v) for j, v in enumerate(flux)})
        rows.append(row)
        print('✓')

    except Exception as e:
        print(f'FAILED - {e}')

# Force exact exoTest column order even when rows are empty
df = pd.DataFrame(rows, columns=ref_columns)
generated_columns = df.columns.tolist()

same_cols_as_test = generated_columns == ref_columns
same_cols_as_train = generated_columns == train_columns

print(f'Generated columns match exoTest: {same_cols_as_test}')
print(f'Generated columns match exoTrain: {same_cols_as_train}')

if not same_cols_as_test:
    missing = [c for c in ref_columns if c not in generated_columns]
    extra = [c for c in generated_columns if c not in ref_columns]
    raise ValueError(f'Column mismatch with exoTest. Missing: {missing}, Extra: {extra}')

df.to_csv(OUTPUT_CSV, index=False)
print(f'\nSaved {len(df)} rows -> {OUTPUT_CSV}  |  shape: {df.shape}')

[1/49] KIC 5531576 ... ✓
[2/49] KIC 5531576 ... ✓
[3/49] KIC 6291653 ... ✓
[4/49] KIC 6392727 ... ✓
[5/49] KIC 6422070 ... ✓
[6/49] KIC 6428700 ... ✓
[7/49] KIC 6428700 ... ✓
[8/49] KIC 5792202 ... ✓
[9/49] KIC 5794379 ... ✓
[10/49] KIC 5794379 ... ✓
[11/49] KIC 5881688 ... ✓
[12/49] KIC 6022556 ... ✓
[13/49] KIC 6032497 ... ✓
[14/49] KIC 6061119 ... ✓
[15/49] KIC 6191521 ... FAILED - ("Connection broken: ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None)", ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))
[16/49] KIC 3531558 ... ✓
[17/49] KIC 6276477 ... ✓
[18/49] KIC 6863998 ... ✓
[19/49] KIC 6948054 ... FAILED - HTTPSConnectionPool(host='mast.stsci.edu', port=443): Max retries exceeded with url: /api/v0.1/Download/file?uri=mast:KEPLER/url/missions/kepler/lightcurves/0069/006948054/kplr006948054-2012088054726_llc.fits (Caused by SSLError(SSLEOFError(8, '[SSL: UN

In [ ]:
# Preview + post-save header validation
saved_columns = pd.read_csv(OUTPUT_CSV, nrows=0).columns.tolist()

print('Saved columns == exoTest columns:', saved_columns == ref_columns)
print('Saved columns == exoTrain columns:', saved_columns == train_columns)

preview_cols = ['LABEL'] + [f'FLUX.{i}' for i in range(1, 6)]
print(df[preview_cols].head(5).to_string(index=False))
print('\nLabel counts:', df['LABEL'].value_counts().to_dict())

Saved columns == exoTest columns: True
Saved columns == exoTrain columns: True
 LABEL   FLUX.1   FLUX.2   FLUX.3   FLUX.4   FLUX.5
     2 1.000168 1.000366 0.999409 0.999980 0.999331
     2 1.000168 1.000366 0.999409 0.999980 0.999331
     2 1.000581 1.001158 1.000151 1.002052 1.000714
     2 1.000581 1.001158 1.000151 1.002052 1.000714
     2 1.004298 1.004899 1.004018 1.004356 1.006037

Label counts: {2: 49}
